# Briefing Continuity A/B Test
<!-- Created: 2026-02-28 -->

**A**: Current prompt (no previous briefing context)  
**C**: Same prompt + yesterday's chapter summaries injected  

Goal: Does providing yesterday's topics improve story continuity and reduce redundancy?

In [2]:
import os, time, json, re
from pathlib import Path
from datetime import date, timedelta
from collections import Counter
from dotenv import load_dotenv
from supabase import create_client
from google import genai
from google.genai import types

env_path = Path('../.env.local') if Path('../.env.local').exists() else Path('.env.local')
load_dotenv(env_path)

sb = create_client(os.getenv('NEXT_PUBLIC_SUPABASE_URL'), os.getenv('SUPABASE_SERVICE_ROLE_KEY'))
gemini_client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

print('Connected: Supabase, Gemini')

Connected: Supabase, Gemini


## Fetch Yesterday's Briefing + Extract Chapter Summaries

In [3]:
today = date.today()
yesterday = today - timedelta(days=1)

# Fetch yesterday's EN briefing from DB
prev_briefing = sb.table('wsj_briefings') \
    .select('briefing_text, chapters, date') \
    .eq('category', 'EN') \
    .eq('date', str(yesterday)) \
    .limit(1) \
    .execute()

if not prev_briefing.data:
    # Try 2 days ago (in case yesterday had no briefing)
    day_before = today - timedelta(days=2)
    prev_briefing = sb.table('wsj_briefings') \
        .select('briefing_text, chapters, date') \
        .eq('category', 'EN') \
        .eq('date', str(day_before)) \
        .limit(1) \
        .execute()

if prev_briefing.data:
    prev = prev_briefing.data[0]
    prev_text = prev['briefing_text']
    prev_date = prev['date']
    print(f'Found previous briefing: {prev_date} ({len(prev_text)} chars)')
else:
    prev_text = None
    prev_date = None
    print('No previous briefing found — will skip C variant')

Found previous briefing: 2026-03-03 (10585 chars)


In [4]:
# Extract chapter titles + first sentence from previous briefing
def extract_chapter_summaries(text: str) -> str:
    """Parse [CHAPTER: Title] markers and grab the first sentence after each.
    If markers were already stripped, fall back to paragraph-based extraction."""
    
    chapter_re = re.compile(r'\[CHAPTER:\s*(.+?)\]\s*\n?', re.MULTILINE)
    matches = list(chapter_re.finditer(text))
    
    if matches:
        # Has chapter markers
        summaries = []
        for i, m in enumerate(matches):
            title = m.group(1).strip()
            # Get text between this marker and next (or end)
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            chunk = text[start:end].strip()
            # First sentence
            first_sentence = chunk.split('. ')[0].strip()
            if first_sentence and not first_sentence.endswith('.'):
                first_sentence += '.'
            if title.lower() not in ('opening', 'market snapshot'):
                summaries.append(f'- {title}: {first_sentence}')
        return '\n'.join(summaries)
    
    # Fallback: no chapter markers (already cleaned text)
    # Split into paragraphs, take first sentence of substantial ones
    paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 100]
    summaries = []
    for p in paragraphs[1:7]:  # Skip opening greeting, take up to 6
        first_sentence = p.split('. ')[0].strip()
        if first_sentence and not first_sentence.endswith('.'):
            first_sentence += '.'
        summaries.append(f'- {first_sentence}')
    return '\n'.join(summaries)

if prev_text:
    yesterday_context = extract_chapter_summaries(prev_text)
    print(f'Extracted context ({len(yesterday_context)} chars, ~{len(yesterday_context)//4} tokens):\n')
    print(yesterday_context)
else:
    yesterday_context = None

Extracted context (464 chars, ~116 tokens):

- Let's get right into it.
- The response from Tehran has been swift and widespread.
- The market reaction has been exactly what you would expect when this much uncertainty is injected into the system.
- The International Monetary Fund, or IMF, has weighed in, stating the obvious: these developments make the global economic outlook far more uncertain.
- And that brings us to the heart of the economic fallout: energy.
- The impact has been immediate and severe.


## Fetch Today's Articles (reuse existing pipeline logic)

In [5]:
# 48h lookback window
cutoff = today - timedelta(hours=48)

items = sb.table('wsj_items') \
    .select('id,title,description,feed_name,published_at,link,briefed') \
    .gte('published_at', cutoff.strftime('%Y-%m-%dT%H:%M:%S')) \
    .order('published_at', desc=True) \
    .execute()

print(f'Articles in window (48h): {len(items.data)}')
cats = Counter(i['feed_name'] for i in items.data)
for cat, count in cats.most_common():
    print(f'  {cat}: {count}')

Articles in window (48h): 188
  BUSINESS_MARKETS: 100
  WORLD: 30
  POLITICS: 24
  ECONOMY: 20
  TECH: 14


In [6]:
# Join with crawl results + LLM analysis
item_ids = [i['id'] for i in items.data]

crawl_map = {}
for i in range(0, len(item_ids), 100):
    batch = item_ids[i:i+100]
    crawls = sb.table('wsj_crawl_results') \
        .select('id,wsj_item_id,content,relevance_flag,relevance_score,llm_same_event,top_image') \
        .in_('wsj_item_id', batch) \
        .eq('relevance_flag', 'ok') \
        .execute()
    for c in crawls.data:
        crawl_map[c['wsj_item_id']] = c

llm_map = {}
crawl_ids = [c['id'] for c in crawl_map.values()]
for i in range(0, len(crawl_ids), 100):
    batch = crawl_ids[i:i+100]
    llms = sb.table('wsj_llm_analysis') \
        .select('crawl_result_id,summary,key_entities,key_numbers,event_type') \
        .in_('crawl_result_id', batch) \
        .execute()
    for l in llms.data:
        llm_map[l['crawl_result_id']] = l

print(f'Crawl results: {len(crawl_map)}')
print(f'LLM analyses: {len(llm_map)}')

Crawl results: 172
LLM analyses: 152


In [7]:
# Assemble articles (simplified — no curation step for test)
RELEVANCE_THRESHOLD = 0.6
articles = []

for item in items.data:
    wid = item['id']
    crawl = crawl_map.get(wid)
    llm = {}
    if crawl and crawl.get('id') in llm_map:
        llm = llm_map[crawl['id']]

    has_quality = False
    if crawl:
        score = crawl.get('relevance_score') or 0
        same_event = crawl.get('llm_same_event', False)
        has_quality = score >= RELEVANCE_THRESHOLD or same_event

    summary = llm.get('summary') or ''
    full_content = (crawl.get('content') or '') if crawl else ''

    base = {
        'title': item['title'],
        'description': item.get('description') or '',
        'category': item['feed_name'],
        'published_at': item.get('published_at', ''),
        'key_entities': llm.get('key_entities', []),
        'key_numbers': [str(n) for n in llm.get('key_numbers', [])],
    }

    if has_quality and summary:
        base['content'] = summary
    elif has_quality and full_content:
        base['content'] = full_content[:800]
    else:
        base['content'] = base['description'] or base['title']

    articles.append(base)

print(f'Assembled {len(articles)} articles')

Assembled 188 articles


## Build Prompts: A (baseline) vs C (with yesterday's context)

In [8]:
# Import the system prompt from the production script
import sys
sys.path.insert(0, '../scripts')
from importlib import import_module
gen = import_module('8_generate_briefing')
SYSTEM_PROMPT = gen.BRIEFING_SYSTEM_EN

# Format articles text
def format_article(a):
    parts = [f"[{a['category']}] {a['title']}"]
    if a.get('published_at'):
        parts.append(f"  Published: {a['published_at'][:16].replace('T', ' ')}")
    if a.get('description'): parts.append(f"  Desc: {a['description']}")
    if a.get('content'): parts.append(f"  Content: {a['content']}")
    if a.get('key_entities'): parts.append(f"  Entities: {', '.join(a['key_entities'])}")
    if a.get('key_numbers'): parts.append(f"  Numbers: {', '.join(a['key_numbers'])}")
    return '\n'.join(parts)

today_str = today.strftime('%A, %B %d, %Y')
articles_text = f"Date: {today_str}\nToday's articles ({len(articles)} total):\n\n"
articles_text += '\n\n'.join(format_article(a) for a in articles)

# --- Prompt A: baseline (no context) ---
prompt_a = SYSTEM_PROMPT + '\n\n' + articles_text

# --- Prompt C: with yesterday's chapter summaries ---
if yesterday_context:
    context_block = (
        f"\n\nYesterday's briefing ({prev_date}) covered these topics:\n"
        f"{yesterday_context}\n\n"
        f"Use this for continuity — connect follow-up stories naturally "
        f"(e.g., 'As we discussed yesterday...'). "
        f"Avoid repeating the same coverage depth for stories already covered in detail. "
        f"Focus your depth on NEW developments today."
    )
    prompt_c = SYSTEM_PROMPT + context_block + '\n\n' + articles_text
else:
    prompt_c = None

print(f'Prompt A: {len(prompt_a)//4:,} tokens (est)')
if prompt_c:
    print(f'Prompt C: {len(prompt_c)//4:,} tokens (est)')
    print(f'Context overhead: +{(len(prompt_c) - len(prompt_a))//4:,} tokens')
else:
    print('Prompt C: skipped (no previous briefing)')

Prompt A: 62,880 tokens (est)
Prompt C: 63,067 tokens (est)
Context overhead: +187 tokens


## Generate A vs C

In [9]:
THINKING_BUDGET = 4096
MODEL = 'gemini-2.5-pro'

def generate(prompt: str, label: str) -> dict:
    print(f'Generating [{label}]...', end=' ', flush=True)
    start = time.time()
    resp = gemini_client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=8192,
            temperature=0.6,
            thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET),
        ),
    )
    elapsed = time.time() - start
    usage = resp.usage_metadata
    thinking = getattr(usage, 'thoughts_token_count', None) or getattr(usage, 'thinking_token_count', 0) or 0
    
    result = {
        'text': resp.text,
        'elapsed': elapsed,
        'input_tokens': usage.prompt_token_count or 0,
        'output_tokens': usage.candidates_token_count or 0,
        'thinking_tokens': thinking,
        'words': len(resp.text.split()),
    }
    print(f'{result["words"]} words, {elapsed:.1f}s (in:{result["input_tokens"]:,} out:{result["output_tokens"]:,} think:{thinking:,})')
    return result

result_a = generate(prompt_a, 'A: baseline')
if prompt_c:
    result_c = generate(prompt_c, 'C: with context')
else:
    result_c = None
    print('Skipping C — no previous briefing context available')

Generating [A: baseline]... 1522 words, 47.3s (in:63,480 out:1,983 think:3,024)
Generating [C: with context]... 1589 words, 40.7s (in:63,645 out:2,062 think:2,206)


## Compare Results

In [10]:
# Cost calculation
COST_PER_1M = {'input': 1.25, 'output': 10.0, 'thinking': 5.0}

def calc_cost(r):
    return (
        r['input_tokens'] * COST_PER_1M['input'] / 1e6 +
        r['output_tokens'] * COST_PER_1M['output'] / 1e6 +
        r['thinking_tokens'] * COST_PER_1M['thinking'] / 1e6
    )

print('=== COMPARISON ===')
print(f'{"":>20} {"A (baseline)":>15} {"C (context)":>15}')
print(f'{"Words":>20} {result_a["words"]:>15,}', end='')
if result_c: print(f' {result_c["words"]:>15,}')
else: print(' —')
print(f'{"Time (s)":>20} {result_a["elapsed"]:>15.1f}', end='')
if result_c: print(f' {result_c["elapsed"]:>15.1f}')
else: print(' —')
print(f'{"Input tokens":>20} {result_a["input_tokens"]:>15,}', end='')
if result_c: print(f' {result_c["input_tokens"]:>15,}')
else: print(' —')
print(f'{"Output tokens":>20} {result_a["output_tokens"]:>15,}', end='')
if result_c: print(f' {result_c["output_tokens"]:>15,}')
else: print(' —')
print(f'{"Cost ($)":>20} {calc_cost(result_a):>15.4f}', end='')
if result_c: print(f' {calc_cost(result_c):>15.4f}')
else: print(' —')

if result_c:
    extra_cost = calc_cost(result_c) - calc_cost(result_a)
    print(f'\nContext overhead cost: ${extra_cost:.4f} per briefing (${extra_cost*30:.2f}/month)')

=== COMPARISON ===
                        A (baseline)     C (context)
               Words           1,522           1,589
            Time (s)            47.3            40.7
        Input tokens          63,480          63,645
       Output tokens           1,983           2,062
            Cost ($)          0.1143          0.1112

Context overhead cost: $-0.0031 per briefing ($-0.09/month)


In [11]:
# Print both briefings for manual comparison
print('=' * 80)
print('BRIEFING A (baseline)')
print('=' * 80)
print(result_a['text'][:3000])
print('\n... [truncated] ...\n')

if result_c:
    print('=' * 80)
    print('BRIEFING C (with yesterday context)')
    print('=' * 80)
    print(result_c['text'][:3000])
    print('\n... [truncated] ...')

BRIEFING A (baseline)
[CHAPTER: Opening]
Good morning, and welcome to the podcast. It is Wednesday, March 04, 2026. The big story, and it is a massive one, is the escalating conflict in the Middle East, which sent shockwaves through every corner of the global market. We will unpack all of that. We also have some major drama shaking up the world of artificial intelligence and some political primary results out of Texas that are making waves.

[CHAPTER: Global Conflict and Market Shock]
Let's get right to it. The situation in the Middle East has escalated dramatically. A joint U.S. and Israeli military campaign against Iran has begun, and commercial satellite imagery confirms strikes have hit central Tehran, resulting in the death of Supreme Leader Ayatollah Ali Khamenei and a large number of other high-ranking Iranian officials. President Trump has dubbed the campaign "Operation Epic Fury," stating its goals are to destroy Iran's missile and naval capabilities and prevent it from acquir

In [12]:
# Save full outputs for detailed review
output_dir = Path('../notebooks/tts_outputs/text')
output_dir.mkdir(parents=True, exist_ok=True)

today_file = today.strftime('%Y-%m-%d')

with open(output_dir / f'ab-test-A-{today_file}.txt', 'w') as f:
    f.write(result_a['text'])
print(f'Saved: ab-test-A-{today_file}.txt')

if result_c:
    with open(output_dir / f'ab-test-C-{today_file}.txt', 'w') as f:
        f.write(result_c['text'])
    print(f'Saved: ab-test-C-{today_file}.txt')

# Save test metadata
meta = {
    'date': str(today),
    'prev_briefing_date': prev_date,
    'article_count': len(articles),
    'yesterday_context_chars': len(yesterday_context) if yesterday_context else 0,
    'A': {k: v for k, v in result_a.items() if k != 'text'},
}
if result_c:
    meta['C'] = {k: v for k, v in result_c.items() if k != 'text'}

with open(output_dir / f'ab-test-meta-{today_file}.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Saved: ab-test-meta-{today_file}.json')

Saved: ab-test-A-2026-03-04.txt
Saved: ab-test-C-2026-03-04.txt
Saved: ab-test-meta-2026-03-04.json


## D Variant: Full Previous Briefing as "Memory"
<!-- Added: 2026-03-04 -->

**D**: Full previous briefing text injected as LLM memory — model uses it for trend/pattern inference, NOT for repetition.

Hypothesis: Giving the full context lets the LLM naturally detect ongoing story arcs and evolving trends, producing a briefing that feels like a host who *remembers* yesterday's show.

In [13]:
# --- Prompt D: full previous briefing as "memory" ---
if prev_text:
    memory_block = (
        f"\n\n--- YOUR MEMORY (previous briefing from {prev_date}) ---\n"
        f"{prev_text}\n"
        f"--- END MEMORY ---\n\n"
        f"The above is your previous briefing. Treat this as your memory — use it to:\n"
        f"- Identify ongoing story arcs and evolving trends\n"
        f"- Recognize which stories are developing vs. one-off events\n"
        f"- Provide richer context when a story has been building over days\n"
        f"- Naturally reference momentum (e.g., 'this is now the third day of...', "
        f"'building on what we saw yesterday...')\n\n"
        f"Do NOT repeat or summarize yesterday's briefing. Your audience already heard it.\n"
        f"Focus entirely on TODAY's new developments, but with the depth and awareness "
        f"that comes from understanding the bigger picture."
    )
    prompt_d = SYSTEM_PROMPT + memory_block + '\n\n' + articles_text
    print(f'Prompt D: {len(prompt_d)//4:,} tokens (est)')
    print(f'Memory overhead: +{(len(prompt_d) - len(prompt_a))//4:,} tokens vs A')
    print(f'Previous briefing size: ~{len(prev_text)//4:,} tokens')
else:
    prompt_d = None
    print('Prompt D: skipped (no previous briefing)')

Prompt D: 65,690 tokens (est)
Memory overhead: +2,809 tokens vs A
Previous briefing size: ~2,646 tokens


In [14]:
# Generate D variant
if prompt_d:
    result_d = generate(prompt_d, 'D: memory')
else:
    result_d = None
    print('Skipping D — no previous briefing')

Generating [D: memory]... 1653 words, 47.9s (in:65,742 out:2,090 think:2,882)


## Compare A vs C vs D

In [15]:
# Updated comparison table: A vs C vs D
print('=== COMPARISON: A vs C vs D ===')
headers = ['A (baseline)', 'C (chapter summary)', 'D (full memory)']
results = [result_a, result_c, result_d]

print(f'{"":>20}', end='')
for h in headers:
    print(f' {h:>20}', end='')
print()

for metric in ['words', 'elapsed', 'input_tokens', 'output_tokens', 'thinking_tokens']:
    label = {'words': 'Words', 'elapsed': 'Time (s)', 'input_tokens': 'Input tokens',
             'output_tokens': 'Output tokens', 'thinking_tokens': 'Think tokens'}[metric]
    print(f'{label:>20}', end='')
    for r in results:
        if r:
            v = r[metric]
            if metric == 'elapsed':
                print(f' {v:>20.1f}', end='')
            else:
                print(f' {v:>20,}', end='')
        else:
            print(f' {"—":>20}', end='')
    print()

print(f'{"Cost ($)":>20}', end='')
for r in results:
    if r:
        print(f' {calc_cost(r):>20.4f}', end='')
    else:
        print(f' {"—":>20}', end='')
print()

if result_d:
    extra = calc_cost(result_d) - calc_cost(result_a)
    print(f'\nD memory overhead: ${extra:.4f}/briefing (${extra*30:.2f}/month)')

=== COMPARISON: A vs C vs D ===
                             A (baseline)  C (chapter summary)      D (full memory)
               Words                1,522                1,589                1,653
            Time (s)                 47.3                 40.7                 47.9
        Input tokens               63,480               63,645               65,742
       Output tokens                1,983                2,062                2,090
        Think tokens                3,024                2,206                2,882
            Cost ($)               0.1143               0.1112               0.1175

D memory overhead: $0.0032/briefing ($0.10/month)


In [16]:
# Print D briefing for manual review
if result_d:
    print('=' * 80)
    print('BRIEFING D (full memory)')
    print('=' * 80)
    print(result_d['text'])
else:
    print('D variant not generated')

BRIEFING D (full memory)
[CHAPTER: Opening]
Good morning, and welcome to the podcast. It is Wednesday,March 4, 2026.

The world is grappling with a rapidly widening conflict in the Middle East and what that means for energy prices, inflation, and global markets. We will break down the latest developments there, including a tragic friendly-fire incident. We will also cover some major political upsets from the Texas primaries and a deeply disturbing lawsuit that puts a spotlight on the dark side of artificial intelligence.

[CHAPTER: The Widening Conflict]
Let's start in the Middle East, where the conflict between the U.S., Israel, and Iran has entered its fifth day and is escalating rapidly. Iran has continued to launch missile and drone attacks on Gulf nations, a significant retaliation for the U.S. and Israeli strikes that killed Supreme Leader Ayatollah Ali Khamenei and other senior officials over the weekend. President Trump has said he expects the operation, dubbed "Epic Fury," cou

In [17]:
# Save D output
if result_d:
    with open(output_dir / f'ab-test-D-{today_file}.txt', 'w') as f:
        f.write(result_d['text'])
    print(f'Saved: ab-test-D-{today_file}.txt')

    # Update metadata
    meta['D'] = {k: v for k, v in result_d.items() if k != 'text'}
    meta['prev_briefing_chars'] = len(prev_text) if prev_text else 0
    with open(output_dir / f'ab-test-meta-{today_file}.json', 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'Updated: ab-test-meta-{today_file}.json')

Saved: ab-test-D-2026-03-04.txt
Updated: ab-test-meta-2026-03-04.json


## F Variant: 2-Day Memory (-1 and -2)
<!-- Added: 2026-03-04 -->

**F**: Full briefing text from **both yesterday AND 2 days ago** as memory.

Hypothesis: Multi-day memory enables detection of 3+ day story arcs and richer trend awareness.

In [18]:
# Fetch 2-day briefing history
day_minus_1 = today - timedelta(days=1)
day_minus_2 = today - timedelta(days=2)

def fetch_briefing(target_date):
    """Fetch EN briefing for a given date, fallback to day before."""
    resp = sb.table('wsj_briefings') \
        .select('briefing_text, date') \
        .eq('category', 'EN') \
        .eq('date', str(target_date)) \
        .limit(1) \
        .execute()
    if resp.data:
        return resp.data[0]
    return None

brief_d1 = fetch_briefing(day_minus_1)
brief_d2 = fetch_briefing(day_minus_2)

if brief_d1:
    print(f'Day -1: {brief_d1["date"]} ({len(brief_d1["briefing_text"])} chars)')
else:
    print('Day -1: not found')

if brief_d2:
    print(f'Day -2: {brief_d2["date"]} ({len(brief_d2["briefing_text"])} chars)')
else:
    print('Day -2: not found')

Day -1: 2026-03-03 (10585 chars)
Day -2: 2026-03-02 (9966 chars)


In [20]:
# --- Prompt F: 2-day memory ---
if brief_d1 and brief_d2:
    memory_block_f = (
        f"\n\n--- YOUR MEMORY (2-day briefing history) ---\n"
        f"\n### Briefing from {brief_d2['date']} (2 days ago):\n"
        f"{brief_d2['briefing_text']}\n"
        f"\n### Briefing from {brief_d1['date']} (yesterday):\n"
        f"{brief_d1['briefing_text']}\n"
        f"--- END MEMORY ---\n\n"
        f"The above are your previous two briefings. Treat these as your memory — use them to:\n"
        f"- Identify ongoing story arcs and how they have evolved over multiple days\n"
        f"- Recognize which stories are gaining momentum vs. fading\n"
        f"- Provide richer context when a story has been building (e.g., 'this is now the third day of...')\n"
        f"- Notice new developments in stories your audience has been following\n\n"
        f"Do NOT repeat or summarize previous briefings. Your audience already heard them.\n"
        f"Focus entirely on TODAY's new developments, but with the depth and awareness "
        f"that comes from understanding the bigger picture."
    )
    prompt_f = SYSTEM_PROMPT + memory_block_f + '\n\n' + articles_text
    print(f'Prompt F: {len(prompt_f)//4:,} tokens (est)')
    print(f'Memory overhead vs A: +{(len(prompt_f) - len(prompt_a))//4:,} tokens')
    print(f'Memory overhead vs D: +{(len(prompt_f) - len(prompt_d))//4:,} tokens')
elif brief_d1:
    print('Only 1 day of history — F would be same as D, skipping')
    prompt_f = None
else:
    print('No previous briefings found — skipping F')
    prompt_f = None

Prompt F: 68,206 tokens (est)
Memory overhead vs A: +5,325 tokens
Memory overhead vs D: +2,516 tokens


In [21]:
# Generate F variant
if prompt_f:
    result_f = generate(prompt_f, 'F: 2-day memory')
else:
    result_f = None
    print('Skipping F')

Generating [F: 2-day memory]... 

1730 words, 43.0s (in:67,968 out:2,152 think:2,384)


## Compare All Variants: A vs C vs D vs F

In [22]:
# Full comparison table
print('=== COMPARISON: A vs C vs D vs F ===')
labels = ['A (baseline)', 'C (chapters)', 'D (1-day mem)', 'F (2-day mem)']
all_results = [result_a, result_c, result_d, result_f]

print(f'{"":>20}', end='')
for h in labels:
    print(f' {h:>16}', end='')
print()

for metric in ['words', 'elapsed', 'input_tokens', 'output_tokens', 'thinking_tokens']:
    label = {'words': 'Words', 'elapsed': 'Time (s)', 'input_tokens': 'Input tokens',
             'output_tokens': 'Output tokens', 'thinking_tokens': 'Think tokens'}[metric]
    print(f'{label:>20}', end='')
    for r in all_results:
        if r:
            v = r[metric]
            if metric == 'elapsed':
                print(f' {v:>16.1f}', end='')
            else:
                print(f' {v:>16,}', end='')
        else:
            print(f' {"—":>16}', end='')
    print()

print(f'{"Cost ($)":>20}', end='')
for r in all_results:
    if r:
        print(f' {calc_cost(r):>16.4f}', end='')
    else:
        print(f' {"—":>16}', end='')
print()

if result_f:
    extra_d = calc_cost(result_d) - calc_cost(result_a)
    extra_f = calc_cost(result_f) - calc_cost(result_a)
    print(f'\nD overhead vs A: ${extra_d:.4f}/briefing (${extra_d*30:.2f}/month)')
    print(f'F overhead vs A: ${extra_f:.4f}/briefing (${extra_f*30:.2f}/month)')

=== COMPARISON: A vs C vs D vs F ===
                         A (baseline)     C (chapters)    D (1-day mem)    F (2-day mem)
               Words            1,522            1,589            1,653            1,730
            Time (s)             47.3             40.7             47.9             43.0
        Input tokens           63,480           63,645           65,742           67,968
       Output tokens            1,983            2,062            2,090            2,152
        Think tokens            3,024            2,206            2,882            2,384
            Cost ($)           0.1143           0.1112           0.1175           0.1184

D overhead vs A: $0.0032/briefing ($0.10/month)
F overhead vs A: $0.0041/briefing ($0.12/month)


In [23]:
# Print F briefing for manual review
if result_f:
    print('=' * 80)
    print('BRIEFING F (2-day memory)')
    print('=' * 80)
    print(result_f['text'])
else:
    print('F variant not generated')

BRIEFING F (2-day memory)
[CHAPTER: Opening]
Good morning, and welcome to the podcast. It is Wednesday, March 4, 2026. The aftershocks from the weekend’s military strikes in the Middle East are getting stronger, not weaker, and today we are going to walk through what that means for the global economy. We will also cover a major political shakeup in Texas that saw a rising star get voted out, and a truly disturbing lawsuit involving an AI chatbot that has taken a tragic turn.

[CHAPTER: The Widening War]
Alright, let's start with the main story that is driving everything else. The conflict in the Middle East is now in its fifth day and is escalating. Iran has reportedly carried out a second consecutive day of drone and missile strikes across the Gulf region. This is in retaliation for the ongoing U.S. and Israeli campaign that has targeted Iran's leadership and military infrastructure. The conflict has now fully drawn in Gulf allies, with reports of damage in the UAE and other nations t

In [24]:
# Save F output + update metadata
if result_f:
    with open(output_dir / f'ab-test-F-{today_file}.txt', 'w') as f:
        f.write(result_f['text'])
    print(f'Saved: ab-test-F-{today_file}.txt')

    meta['F'] = {k: v for k, v in result_f.items() if k != 'text'}
    if brief_d2:
        meta['prev_briefing_d2_chars'] = len(brief_d2['briefing_text'])
        meta['prev_briefing_d2_date'] = brief_d2['date']
    with open(output_dir / f'ab-test-meta-{today_file}.json', 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'Updated: ab-test-meta-{today_file}.json')

Saved: ab-test-F-2026-03-04.txt
Updated: ab-test-meta-2026-03-04.json


## G Variant: 3-Day Memory (-1, -2, -3)
<!-- Added: 2026-03-04 -->

**G**: Full briefing text from **3 days** (-1, -2, -3) as memory.

Hypothesis: Does a third day add meaningful arc detection, or is it diminishing returns?

In [25]:
# Fetch 3-day briefing history
day_minus_3 = today - timedelta(days=3)

brief_d3 = fetch_briefing(day_minus_3)
if not brief_d3:
    # Try day -4 fallback
    brief_d3 = fetch_briefing(today - timedelta(days=4))

if brief_d3:
    print(f'Day -3: {brief_d3["date"]} ({len(brief_d3["briefing_text"])} chars)')
else:
    print('Day -3: not found')

# Recap all 3
for label, b in [('Day -1', brief_d1), ('Day -2', brief_d2), ('Day -3', brief_d3)]:
    if b:
        print(f'  {label}: {b["date"]} ({len(b["briefing_text"])} chars, ~{len(b["briefing_text"])//4} tokens)')
    else:
        print(f'  {label}: missing')

total_mem = sum(len(b['briefing_text']) for b in [brief_d1, brief_d2, brief_d3] if b)
print(f'\nTotal memory: {total_mem:,} chars (~{total_mem//4:,} tokens)')

Day -3: 2026-03-01 (10607 chars)
  Day -1: 2026-03-03 (10585 chars, ~2646 tokens)
  Day -2: 2026-03-02 (9966 chars, ~2491 tokens)
  Day -3: 2026-03-01 (10607 chars, ~2651 tokens)

Total memory: 31,158 chars (~7,789 tokens)


In [26]:
# --- Prompt G: 3-day memory ---
if brief_d1 and brief_d2 and brief_d3:
    memory_block_g = (
        f"\n\n--- YOUR MEMORY (3-day briefing history) ---\n"
        f"\n### Briefing from {brief_d3['date']} (3 days ago):\n"
        f"{brief_d3['briefing_text']}\n"
        f"\n### Briefing from {brief_d2['date']} (2 days ago):\n"
        f"{brief_d2['briefing_text']}\n"
        f"\n### Briefing from {brief_d1['date']} (yesterday):\n"
        f"{brief_d1['briefing_text']}\n"
        f"--- END MEMORY ---\n\n"
        f"The above are your previous three briefings. Treat these as your memory — use them to:\n"
        f"- Identify ongoing story arcs and how they have evolved over multiple days\n"
        f"- Recognize which stories are gaining momentum, fading, or entering a new phase\n"
        f"- Provide richer context when a story has been building (e.g., 'this is now the fourth day of...')\n"
        f"- Notice new developments in stories your audience has been following\n\n"
        f"Do NOT repeat or summarize previous briefings. Your audience already heard them.\n"
        f"Focus entirely on TODAY's new developments, but with the depth and awareness "
        f"that comes from understanding the bigger picture."
    )
    prompt_g = SYSTEM_PROMPT + memory_block_g + '\n\n' + articles_text
    print(f'Prompt G: {len(prompt_g)//4:,} tokens (est)')
    print(f'Memory overhead vs A: +{(len(prompt_g) - len(prompt_a))//4:,} tokens')
    print(f'Memory overhead vs F: +{(len(prompt_g) - len(prompt_f))//4:,} tokens')
else:
    missing = [l for l, b in [('-1', brief_d1), ('-2', brief_d2), ('-3', brief_d3)] if not b]
    print(f'Missing briefings for day(s): {missing} — skipping G')
    prompt_g = None

Prompt G: 70,875 tokens (est)
Memory overhead vs A: +7,994 tokens
Memory overhead vs F: +2,669 tokens


In [27]:
# Generate G variant
if prompt_g:
    result_g = generate(prompt_g, 'G: 3-day memory')
else:
    result_g = None
    print('Skipping G')

Generating [G: 3-day memory]... 1472 words, 42.1s (in:70,238 out:1,834 think:2,437)


In [28]:
# Full comparison: A vs D vs F vs G
print('=== COMPARISON: A vs D vs F vs G ===')
labels = ['A (baseline)', 'D (1-day)', 'F (2-day)', 'G (3-day)']
all_r = [result_a, result_d, result_f, result_g]

print(f'{"":>20}', end='')
for h in labels:
    print(f' {h:>16}', end='')
print()

for metric in ['words', 'elapsed', 'input_tokens', 'output_tokens', 'thinking_tokens']:
    label = {'words': 'Words', 'elapsed': 'Time (s)', 'input_tokens': 'Input tokens',
             'output_tokens': 'Output tokens', 'thinking_tokens': 'Think tokens'}[metric]
    print(f'{label:>20}', end='')
    for r in all_r:
        if r:
            v = r[metric]
            if metric == 'elapsed':
                print(f' {v:>16.1f}', end='')
            else:
                print(f' {v:>16,}', end='')
        else:
            print(f' {"—":>16}', end='')
    print()

print(f'{"Cost ($)":>20}', end='')
for r in all_r:
    if r:
        print(f' {calc_cost(r):>16.4f}', end='')
    else:
        print(f' {"—":>16}', end='')
print()

for label, r in zip(['D', 'F', 'G'], [result_d, result_f, result_g]):
    if r:
        extra = calc_cost(r) - calc_cost(result_a)
        print(f'{label} overhead vs A: ${extra:.4f}/briefing (${extra*30:.2f}/month)')

=== COMPARISON: A vs D vs F vs G ===
                         A (baseline)        D (1-day)        F (2-day)        G (3-day)
               Words            1,522            1,653            1,730            1,472
            Time (s)             47.3             47.9             43.0             42.1
        Input tokens           63,480           65,742           67,968           70,238
       Output tokens            1,983            2,090            2,152            1,834
        Think tokens            3,024            2,882            2,384            2,437
            Cost ($)           0.1143           0.1175           0.1184           0.1183
D overhead vs A: $0.0032/briefing ($0.10/month)
F overhead vs A: $0.0041/briefing ($0.12/month)
G overhead vs A: $0.0040/briefing ($0.12/month)


In [29]:
# Print G briefing
if result_g:
    print('=' * 80)
    print('BRIEFING G (3-day memory)')
    print('=' * 80)
    print(result_g['text'])
else:
    print('G variant not generated')

BRIEFING G (3-day memory)
[CHAPTER: Opening]
Good morning, and welcome to the podcast. It is Wednesday, March 4, 2026. Thanks for tuning in. The conflict in the Middle East is now entering its fifth day, and the economic fallout is starting to ripple across the globe in some pretty serious ways. We are going to unpack what that means for energy prices and global markets. We will also check in on the drama between the Pentagon and the AI industry, and break down some surprising results from the Texas primary elections that could reshape the political map.

[CHAPTER: Middle East Conflict Widens]
Alright, let us get straight to it. The conflict between the U.S., Israel, and Iran is widening. Iran has reportedly stepped up its missile and drone attacks across the Gulf region for a second straight day. This is drawing in nations that had been trying to stay on the sidelines, with attacks reported in the UAE, Qatar, Saudi Arabia, Kuwait, and Jordan. The human cost is also becoming clearer. T

In [30]:
# Save G output + update metadata
if result_g:
    with open(output_dir / f'ab-test-G-{today_file}.txt', 'w') as f:
        f.write(result_g['text'])
    print(f'Saved: ab-test-G-{today_file}.txt')

    meta['G'] = {k: v for k, v in result_g.items() if k != 'text'}
    if brief_d3:
        meta['prev_briefing_d3_chars'] = len(brief_d3['briefing_text'])
        meta['prev_briefing_d3_date'] = brief_d3['date']
    with open(output_dir / f'ab-test-meta-{today_file}.json', 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'Updated: ab-test-meta-{today_file}.json')

Saved: ab-test-G-2026-03-04.txt
Updated: ab-test-meta-2026-03-04.json


## Temperature Test: F prompt with temp 0.3 vs 0.6
<!-- Added: 2026-03-04 -->

Using the F (2-day memory) prompt, compare temperature 0.3 vs the existing 0.6 result.

In [ ]:
# Generate F with temperature 0.3
def generate_with_temp(prompt: str, label: str, temp: float) -> dict:
    print(f'Generating [{label}] temp={temp}...', end=' ', flush=True)
    import time as _time
    start = _time.time()
    resp = gemini_client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=8192,
            temperature=temp,
            thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET),
        ),
    )
    elapsed = _time.time() - start
    usage = resp.usage_metadata
    thinking = getattr(usage, 'thoughts_token_count', None) or getattr(usage, 'thinking_token_count', 0) or 0
    result = {
        'text': resp.text,
        'elapsed': elapsed,
        'input_tokens': usage.prompt_token_count or 0,
        'output_tokens': usage.candidates_token_count or 0,
        'thinking_tokens': thinking,
        'words': len(resp.text.split()),
        'temperature': temp,
    }
    print(f'{result["words"]} words, {elapsed:.1f}s')
    return result

result_f_03 = generate_with_temp(prompt_f, 'F temp=0.3', 0.3)

In [ ]:
# Compare F@0.6 vs F@0.3
print('=== F TEMPERATURE COMPARISON ===')
print(f'{"":>20} {"F (temp=0.6)":>16} {"F (temp=0.3)":>16}')
for metric in ['words', 'elapsed', 'input_tokens', 'output_tokens', 'thinking_tokens']:
    label = {'words': 'Words', 'elapsed': 'Time (s)', 'input_tokens': 'Input tokens',
             'output_tokens': 'Output tokens', 'thinking_tokens': 'Think tokens'}[metric]
    v1 = result_f[metric]
    v2 = result_f_03[metric]
    if metric == 'elapsed':
        print(f'{label:>20} {v1:>16.1f} {v2:>16.1f}')
    else:
        print(f'{label:>20} {v1:>16,} {v2:>16,}')
print(f'{"Cost ($)":>20} {calc_cost(result_f):>16.4f} {calc_cost(result_f_03):>16.4f}')

In [ ]:
# Print F@0.3 briefing
print('=' * 80)
print('BRIEFING F (2-day memory, temp=0.3)')
print('=' * 80)
print(result_f_03['text'])

In [ ]:
# Save F@0.3 output
with open(output_dir / f'ab-test-F03-{today_file}.txt', 'w') as f:
    f.write(result_f_03['text'])
print(f'Saved: ab-test-F03-{today_file}.txt')

meta['F_temp03'] = {k: v for k, v in result_f_03.items() if k != 'text'}
with open(output_dir / f'ab-test-meta-{today_file}.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Updated: ab-test-meta-{today_file}.json')